## ⚙️ CRUD y operaciones básicas para Usuarios

**Propósito:** Esta sección agrupa las celdas correspondientes a las cuatro operaciones fundamentales de persistencia (**C**reate, **R**ead, **U**pdate, **D**elete) y consultas básicas sobre la colección de usuarios.

In [1]:
# Celda 1 - Conexión e imports para usuarios
from dao.mongo_dao import MongoDAO
from dao.usuario_dao import UsuarioDAO
from bson import ObjectId

mongo = MongoDAO()
db = mongo.connect()
usuario_dao = UsuarioDAO(mongo)
print("Conectado a DB:", mongo.db.name)


Conectado a DB: civictech


### 👤 Crear usuario y mostrar resumen público
Inserta un usuario de prueba validando su dependencia (`municipio_id`). Imprime el resultado de la operación de forma segura, sin exponer credenciales en pantalla.

In [4]:
# Celda: Crear usuario (presentación segura)
import importlib
import dao.usuario_dao as ud
import dao.municipio_dao as md
importlib.reload(ud)
importlib.reload(md)
from dao.usuario_dao import UsuarioDAO
from dao.municipio_dao import MunicipioDAO

# Instanciar DAOs (asume que 'mongo' está disponible en el notebook)
usuario_dao = UsuarioDAO(mongo)
municipio_dao = MunicipioDAO(mongo)

# Ajustá los datos según necesites
nuevo_usuario = {
    "nombre_apellido": "María López",
    "dni": "876545321",
    "email": "maria.lopez21@example.com",
    "contrasena": "MiPasswordSeguro123",
    "municipio_id": "6a28b11e3ca8ed0b349df8a3",
    "telefono": "+54 9 11 0000 0000",
    "reputacion": 0
}

def safe_print_user_summary(u):
    if not u:
        print("No hay datos del usuario para mostrar.")
        return
    print("Resumen usuario (seguro):")
    print("  _id         :", str(u.get("_id")))
    print("  nombre      :", u.get("nombre_apellido"))
    print("  dni         :", u.get("dni"))
    print("  email       :", u.get("email"))
    print("  municipio_id:", str(u.get("municipio_id")))
    print("  telefono    :", u.get("telefono"))
    print("  reputacion  :", u.get("reputacion"))
    print("  rol         :", u.get("rol"))
    print("  created_at  :", u.get("created_at"))

try:
    # Verificar existencia del municipio antes de crear para mensajes más claros
    if not municipio_dao.obtener_por_id(nuevo_usuario["municipio_id"]):
        print("⛔ Municipio indicado no existe. Ajustá 'municipio_id' antes de crear el usuario.")
    else:
        uid = usuario_dao.create_user(nuevo_usuario)
        print("✅ Usuario creado con _id:", uid)
        # Mostrar resumen seguro del usuario (sin contrasena)
        user_safe = usuario_dao.obtener_publico_por_id(uid)
        safe_print_user_summary(user_safe)
except ValueError as e:
    print("⚠️ No se creó el usuario:", e)
except Exception as e:
    import traceback
    print("❌ Error inesperado:", type(e).__name__, "-", e)
    traceback.print_exc()


✅ Usuario creado con _id: 6a28e6f389b62b0c7661fa04
Resumen usuario (seguro):
  _id         : 6a28e6f389b62b0c7661fa04
  nombre      : María López
  dni         : 876545321
  email       : maria.lopez21@example.com
  municipio_id: 6a28b11e3ca8ed0b349df8a3
  telefono    : +54 9 11 0000 0000
  reputacion  : 0
  rol         : usuario
  created_at  : 2026-06-10 04:24:19.950000


In [ ]:
--------


### 🛠️ Operaciones comunes para `usuario_dao` en el notebook

**Propósito:** Agrupa los métodos, consultas y pruebas de uso frecuente para gestionar la colección de usuarios utilizando la capa de persistencia (`usuario_dao`).

In [5]:
# Celda A - recargar módulos e instanciar DAOs
import importlib
import dao.usuario_dao as ud
import dao.municipio_dao as md
importlib.reload(ud)
importlib.reload(md)
from dao.usuario_dao import UsuarioDAO
from dao.municipio_dao import MunicipioDAO

# Asume que 'mongo' está disponible en el notebook
usuario_dao = UsuarioDAO(mongo)
municipio_dao = MunicipioDAO(mongo)

print("DAOs listos: usuario_dao, municipio_dao")


DAOs listos: usuario_dao, municipio_dao


In [16]:
# Celda B - listar usuarios en formato tabla (ID completo; mostrar nombre del municipio en vez de su id; sin columna 'rol')
import importlib
import dao.usuario_dao as ud
import dao.municipio_dao as md
importlib.reload(ud)
importlib.reload(md)
from dao.usuario_dao import UsuarioDAO
from dao.municipio_dao import MunicipioDAO

# Instanciar DAOs (asume que 'mongo' está disponible en el notebook)
usuario_dao = UsuarioDAO(mongo)
municipio_dao = MunicipioDAO(mongo)

users = usuario_dao.list_users(limit=50)

def safe_str(v):
    try:
        return "" if v is None else str(v)
    except Exception:
        return repr(v)

def get_municipio_nombre(mid):
    try:
        if not mid:
            return ""
        doc = municipio_dao.obtener_por_id(mid)
        return doc.get("nombre") if doc else ""
    except Exception:
        return ""

def format_table(rows, headers):
    # calcular anchos por columna (una línea por celda)
    widths = []
    for i, h in enumerate(headers):
        col_vals = [str(r[i]) for r in rows] if rows else []
        max_cell = max([len(v) for v in col_vals], default=0)
        widths.append(max(len(h), max_cell))
    # cabecera
    header_line = " | ".join(h.ljust(widths[i]) for i, h in enumerate(headers))
    sep = "-+-".join("-" * widths[i] for i in range(len(headers)))
    print(header_line)
    print(sep)
    # filas
    for r in rows:
        print(" | ".join(str(r[i]).ljust(widths[i]) for i in range(len(headers))))

if not users:
    print("No se encontraron usuarios.")
else:
    headers = ["_id", "nombre", "dni", "email", "municipio (nombre)", "telefono", "reputacion"]
    rows = []
    for u in users:
        municipio_id = u.get("municipio_id")
        municipio_nombre = get_municipio_nombre(municipio_id)
        rows.append([
            safe_str(u.get("_id")),                      # ID completo
            u.get("nombre_apellido") or "",
            u.get("dni") or "",
            u.get("email") or "",
            municipio_nombre or "",                      # nombre del municipio en lugar del id
            u.get("telefono") or "",
            str(u.get("reputacion") or "")
        ])
    print(f"Mostrando {len(rows)} usuarios (tabla):\n")
    format_table(rows, headers)


Mostrando 13 usuarios (tabla):

_id                      | nombre      | dni       | email                     | municipio (nombre) | telefono           | reputacion
-------------------------+-------------+-----------+---------------------------+--------------------+--------------------+-----------
6a28b11e3ca8ed0b349df8a6 | Juan Perez  | 30111222  | juan.perez@chilecito.com  | Chilecito          |                    | 50        
6a28b11e3ca8ed0b349df8a7 | Maria Gomez | 31111222  | maria.g@chilecito.com     | Chilecito          |                    | 45        
6a28b11e3ca8ed0b349df8a8 | Carlos Ruiz | 32111222  | carlos.r@chilecito.com    | Chilecito          |                    | 60        
6a28b11e3ca8ed0b349df8a9 | Ana Torres  | 33111222  | ana.t@chilecito.com       | Chilecito          |                    | 55        
6a28b11e3ca8ed0b349df8aa | Luis Diaz   | 34111222  | luis.d@larioja.com        | La Rioja Capital   |                    | 50        
6a28b11e3ca8ed0b349df8ab | Mar

In [12]:
# Celda C - Buscar usuario por _id y mostrar sin contrasena
usuario_id = "6a28b11e3ca8ed0b349df8a7"  # ajustar si querés
try:
    user = usuario_dao.obtener_publico_por_id(usuario_id)
    if not user:
        print("Usuario no encontrado para _id:", usuario_id)
    else:
        print("Usuario encontrado (seguro):")
        print("  _id        :", str(user.get("_id")))
        print("  nombre     :", user.get("nombre_apellido"))
        print("  dni        :", user.get("dni"))
        print("  email      :", user.get("email"))
        print("  municipio  :", str(user.get("municipio_id")))
        print("  telefono   :", user.get("telefono"))
        print("  reputacion :", user.get("reputacion"))
        print("  rol        :", user.get("rol"))
        print("  created_at :", user.get("created_at"))
except Exception as e:
    print("Error al obtener usuario:", type(e).__name__, "-", e)


Usuario encontrado (seguro):
  _id        : 6a28b11e3ca8ed0b349df8a7
  nombre     : Maria Gomez
  dni        : 31111222
  email      : maria.g@chilecito.com
  municipio  : 6a28b11e3ca8ed0b349df8a4
  telefono   : None
  reputacion : 45
  rol        : None
  created_at : None


In [13]:
# Celda D - actualizar email de un usuario

usuario_id = "poné_aquí_un_id_valido" #ajustar id

nuevo_email = "nuevo.email@example.com"

try:
    modified = usuario_dao.update_email(usuario_id, nuevo_email)
    if modified:
        print("✅ Email actualizado correctamente.")
        u = usuario_dao.obtener_publico_por_id(usuario_id)
        print("  _id  :", str(u.get("_id")))
        print("  email:", u.get("email"))
    else:
        print("⚠️ No se modificó el documento (modified_count = 0). Verificá si el email ya era el mismo.")
except ValueError as ve:
    print("⛔ No se actualizó:", ve)
except Exception as e:
    print("❌ Error inesperado:", type(e).__name__, "-", e)


⛔ No se actualizó: ID inválido


In [17]:
# Celda G - eliminar usuario por _id (operación destructiva)

#usuario_id = "poné_aquí_un_id_valido"

usuario_id = "6a28b11e3ca8ed0b349df8b1"

try:
    deleted = usuario_dao.delete(usuario_id)
    if deleted:
        print("✅ Usuario eliminado correctamente (deleted_count = 1). _id:", usuario_id)
    else:
        print("⚠️ Intento de borrado realizado pero deleted_count = 0. Verificá si el _id existe.")
except Exception as e:
    print("❌ Error al eliminar usuario:", type(e).__name__, "-", e)


⚠️ Intento de borrado realizado pero deleted_count = 0. Verificá si el _id existe.


In [ ]:
-----